In [28]:
from pathlib import Path
import urllib.request


def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [4]:
print(shakespeare_text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [7]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}

In [9]:
char_to_id["a"]

13

In [13]:
id_to_char[13]

'a'

In [18]:
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [19]:
encoded = encode_text("Hello world!")
encoded

tensor([20, 17, 24, 24, 27,  1, 35, 27, 30, 24, 16,  2])

In [22]:
decoded = decode_text(encoded)
decoded

'hello world!'

In [23]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx + 1:end + 1]
        return window, target

In [25]:
window_length = 50
batch_size = 512
train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

In [27]:
import torch.nn as nn

torch.manual_seed(42)
embed = nn.Embedding(5, 3)
embed(torch.tensor(([[3, 2], [0, 2]])))

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [34]:
class ShakeSpeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
model = ShakeSpeareModel(len(vocab)).to(device)

In [36]:
import torchmetrics
from utils.utils import train

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss().to(device)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)

train(model, optimizer, criterion, train_loader, valid_loader, metric, 10, 3)

Epoch: 1/10, Loss: 1.3796, Val Score: 0.5482
Epoch: 2/10, Loss: 1.3610, Val Score: 0.5508
Epoch: 3/10, Loss: 1.3490, Val Score: 0.5500
Epoch: 4/10, Loss: 1.3404, Val Score: 0.5508
Epoch: 5/10, Loss: 1.3340, Val Score: 0.5515
Epoch: 6/10, Loss: 1.3290, Val Score: 0.5519
Epoch: 7/10, Loss: 1.3250, Val Score: 0.5506
Epoch: 8/10, Loss: 1.3217, Val Score: 0.5507
Epoch: 9/10, Loss: 1.3188, Val Score: 0.5523
Epoch: 10/10, Loss: 1.3165, Val Score: 0.5503


0.5522639155387878

In [38]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]
predicted_char

'e'

In [47]:
torch.manual_seed(42)
probs = torch.tensor([0.5, 0.4, 0.1])
samples = torch.multinomial(probs, replacement=True, num_samples=20)
samples

tensor([0, 0, 0, 0, 1, 0, 2, 2, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0])

In [48]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [49]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [51]:
text_to_extend = "To be or not to b"

In [54]:
print(extend_text(model, text_to_extend, n_chars=80, temperature=0.01))

To be or not to be a prove the seat of the prince,
and see the seat of the prince that shall be s


In [55]:
print(extend_text(model, text_to_extend, n_chars=80, temperature=0.4))

To be or not to be tender should be the more than the company
and warwick shall not see the subje


In [56]:
print(extend_text(model, text_to_extend, n_chars=80, temperature=100))

To be or not to bns3jqt&;&&jb
xq:cu$ u?,tgs,gipq o!m;xydlv; j?eybehf.an!;lvasrf!c f'?yfapmguutz.h
